# Grovarc — LLaMA 3 8B QLoRA Fine-tuning

개발자 주간 회고 초안 생성 모델 학습

- **Base model**: `meta-llama/Meta-Llama-3-8B-Instruct`
- **방식**: 4-bit QLoRA (NF4 + Double Quantization)
- **환경**: Google Colab T4 GPU (16GB VRAM)
- **데이터**: `train.jsonl` / `val.jsonl` (ChatML messages 형식)

## 사전 준비
1. Colab 런타임 → **T4 GPU** 선택
2. Google Drive에 `train.jsonl` / `val.jsonl` 업로드
3. Hugging Face 토큰 준비 (LLaMA 3 접근 권한 필요)


## Step 1 — 패키지 설치

In [ ]:
%%capture
!pip install -q \
    transformers==4.43.0 \
    datasets==2.21.0 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    huggingface_hub

## Step 2 — Google Drive 마운트 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 데이터 경로 설정
DRIVE_BASE = "/content/drive/MyDrive/grovarc-finetune"
TRAIN_FILE = f"{DRIVE_BASE}/train.jsonl"
VAL_FILE   = f"{DRIVE_BASE}/val.jsonl"
OUTPUT_DIR = f"{DRIVE_BASE}/outputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Train: {TRAIN_FILE}")
print(f"Val:   {VAL_FILE}")

## Step 3 — HuggingFace 로그인 및 GPU 확인

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Colab Secrets에 HF_TOKEN 저장 후 사용
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("HuggingFace 로그인 완료")

In [ ]:
import torch

print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")

## Step 4 — 데이터셋 로드 및 포맷 변환

ChatML messages 형식 → LLaMA 3 Instruct 형식으로 변환

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # gradient checkpointing과 호환

# JSONL 데이터셋 로드
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)
print(f"Train: {len(dataset['train'])}개 / Val: {len(dataset['validation'])}개")
print("\n샘플 확인:")
print(dataset['train'][0])

In [ ]:
def format_sample(sample: dict) -> dict:
    """ChatML messages → LLaMA 3 Instruct 채팅 템플릿 적용"""
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_sample, remove_columns=["messages", "source"])

print("포맷 변환 완료. 샘플 텍스트:")
print(dataset['train'][0]['text'][:500])

## Step 5 — 4-bit QLoRA 모델 로드

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — QLoRA 논문 권장
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # 이중 양자화로 추가 메모리 절약
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model.config.use_cache = False           # gradient checkpointing과 충돌 방지
model.config.pretraining_tp = 1

print(f"모델 로드 완료")
print(f"메모리 사용: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 6 — LoRA 어댑터 설정

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit 학습 준비 (gradient checkpointing 활성화)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                  # rank — 높을수록 표현력↑, VRAM↑
    lora_alpha=32,         # scaling = alpha/r = 2
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[       # LLaMA 3 attention + FFN 레이어
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 예상 출력: trainable params: ~20M (전체 ~8B의 0.24%)

## Step 7 — SFTTrainer 설정 및 학습

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # 유효 배치 = 2 * 4 = 8
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.001,
    bf16=True,
    max_grad_norm=0.3,
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    evaluation_strategy="steps",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
)

print("학습 시작...")
trainer.train()
print("학습 완료")

## Step 8 — 학습 곡선 확인

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = [l["step"] for l in logs if "loss" in l]
train_loss  = [l["loss"] for l in logs if "loss" in l]
eval_steps  = [l["step"] for l in logs if "eval_loss" in l]
eval_loss   = [l["eval_loss"] for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_loss, label="Train Loss")
if eval_loss:
    plt.plot(eval_steps, eval_loss, label="Val Loss", linestyle="--")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Grovarc LLaMA 3 QLoRA — Training Curve")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curve.png", dpi=150)
plt.show()

## Step 9 — LoRA 어댑터 저장 및 풀 모델 병합

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/lora-adapter"
MERGED_DIR  = f"{OUTPUT_DIR}/merged-model"

# LoRA 어댑터만 저장
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"어댑터 저장 완료: {ADAPTER_DIR}")

In [ ]:
from peft import AutoPeftModelForCausalLM

# 어댑터 + 베이스 모델 병합 (추론용)
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"병합 모델 저장 완료: {MERGED_DIR}")

## Step 10 — 추론 테스트

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
)

test_messages = [
    {
        "role": "system",
        "content": "당신은 개발자의 성장을 돕는 AI 코치입니다. 주어진 작업 로그를 바탕으로 진솔하고 통찰력 있는 주간 회고를 작성해주세요. 회고는 한국어로 작성하며 마크다운 형식을 사용합니다.",
    },
    {
        "role": "user",
        "content": """## 이번 주 작업 로그 (2026-03-28 ~ 2026-04-03)

[2026-03-28] FastAPI 라우터 구현
성장 코칭 Agent의 /coaching 엔드포인트를 구현했다. LangGraph 5노드 그래프를 연결했다.

[2026-03-30] DuckDuckGo 검색 연동
학습 리소스 검색에 DuckDuckGoSearchRun을 연동했다. Tavily보다 API 키 없이 바로 쓸 수 있어서 좋았다.

[2026-04-02] Fine-tuning 데이터셋 파이프라인 구축
ChatML 형식으로 TrainingSample 스키마를 설계했다. Claude Haiku로 합성 데이터를 생성하는 스크립트도 작성했다.

위 내용을 바탕으로 주간 회고를 작성해주세요.""",
    },
]

output = pipe(
    test_messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

print("=== Fine-tuned 모델 회고 출력 ===")
print(output[0]["generated_text"][-1]["content"])

## 다음 단계

학습 완료 후 진행할 작업:

1. **Hugging Face Hub 업로드** (`#50`)
   ```python
   merged_model.push_to_hub("projectmiluju/grovarc-llama3-8b")
   tokenizer.push_to_hub("projectmiluju/grovarc-llama3-8b")
   ```

2. **AI 서버 연동** (`#51`)
   - `model_service.py`에서 Fine-tuned 모델 로드
   - 환경변수로 Claude ↔ Fine-tuned 모델 A/B 전환
